# Machine Learning Lab: Neural Network Solutions

This notebook contains the complete, debugged, and optimized solutions for Exercise 1 (Boston Housing Regression using Keras) and Exercise 2 (MNIST Digits Classification using PyTorch).

## Exercise (1): Solve a regression task (house price prediction) by learning a neural network

- Define a Sequential or functional model with 5 fully-connected layers having 20, 15, 10, 5, and 1 units (neurons), respectively. Use rectified linear unit (Relu) as activation layer in the first four layers.
- Learn the model using the appropriate loss functions
- Monitor the loss and evaluation metrics
- Compute the number of parameters in your model.

In [ ]:
import torch
import keras
from torchvision import datasets, transforms
import numpy as np

print("--- EXERCISE 1: REGRESSION TASK STARTED ---")

# 1. Loading the Boston Housing dataset from Keras with test_split=0.2
(X_train_reg, Y_train_reg), (X_test_reg, Y_test_reg) = keras.datasets.boston_housing.load_data(
    path="boston_housing.npz", test_split=0.2, seed=113
)
print("Data Shapes -> X_train:", X_train_reg.shape, "Y_train:", Y_train_reg.shape)
print("Data Shapes -> X_test:", X_test_reg.shape, "Y_test:", Y_test_reg.shape)

# 2. Z-score normalization for stable neural network training
mean = X_train_reg.mean(axis=0)
std = X_train_reg.std(axis=0)
X_train_reg = (X_train_reg - mean) / std
X_test_reg = (X_test_reg - mean) / std

# 3. Build 5-layer fully connected sequential network blueprint
model_reg = keras.Sequential([
    keras.layers.Input(shape=(X_train_reg.shape[1],)),
    keras.layers.Dense(20, activation='relu'),
    keras.layers.Dense(15, activation='relu'),
    keras.layers.Dense(10, activation='relu'),
    keras.layers.Dense(5, activation='relu'),
    keras.layers.Dense(1) # Final continuous valuation output
])

# 4. Compile with standard Mean Squared Error loss and mean absolute metric tracking
model_reg.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 5. Print summary to inspect model parameter layout
print("\n--- Regression Model Summary ---")
model_reg.summary()

# 6. Programmatic calculation and verification of explicit parameter allocations
print("\n--- Manual Parameter Allocation Analysis ---")
print(f"Layer 1 (Dense 20): (13 inputs * 20 weights) + 20 biases = {13*20 + 20}")
print(f"Layer 2 (Dense 15): (20 inputs * 15 weights) + 15 biases = {20*15 + 15}")
print(f"Layer 3 (Dense 10): (15 inputs * 10 weights) + 10 biases = {15*10 + 10}")
print(f"Layer 4 (Dense 5):  (10 inputs * 5 weights) + 5 biases  = {10*5 + 5}")
print(f"Layer 5 (Dense 1):  (5 inputs * 1 weight) + 1 bias     = {5*1 + 1}")
total_calculated_params = (13*20+20) + (20*15+15) + (15*10+10) + (10*5+5) + (5*1+1)
print(f"Calculated Total Trainable Weights: {total_calculated_params}")
print(f"Verified Model Param Counter Check: {model_reg.count_params()}\n")

# 7. Train the neural network architecture across 100 historical intervals
print("--- Training Regression Model ---")
history_reg = model_reg.fit(X_train_reg, Y_train_reg, validation_data=(X_test_reg, Y_test_reg), epochs=100, batch_size=32, verbose=1)

## Exercise (2): Solve a classification problem by defining a 5-layers Convolutional neural network

- Let's consider a **Sequential model** with **input shape (28, 28, 1)**, one **Convolutional layer** (32 filters, 5 filter size, and 'Relu' activation), one **Max pooling layer** (2 stride), one **Dropout** (0.3 probability), one **Fully-connected layer** (128 units and 'Relu' activation), and one **Output layer** (10 units and 'Softmax' activation).
- Consider a classification task (e.g., MNIST Digits classification task).
- Learn this model using the appropriate loss functions
- Monitor the loss and evaluation metrics
- Compute the number of parameters in your model

NB: Standard log_softmax and NLLLoss combination is applied for robust mathematical stability.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("\n\n--- EXERCISE 2: CNN CLASSIFICATION TASK STARTED ---")

# 1. Load target tensor datasets directly into workspace tracking frames
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True,
                      transform=transforms.Compose([transforms.ToTensor()]))

mnist_testset = datasets.MNIST(root='./data', train=False, download=True,
                      transform=transforms.Compose([transforms.ToTensor()]))

# 2. Initialize programmatic iterable mini-batch generators
train_dataloader = torch.utils.data.DataLoader(mnist_trainset, batch_size=64, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(mnist_testset, batch_size=64, shuffle=False)

# 3. Build custom modular class strictly adhering to architectural prompts
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        # Layer 1: Conv Layer with 32 filters, 5x5 structural filter size
        self.conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5)
        # Layer 2: Max Pooling with stride=2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Layer 3: Dropout element applying 0.3 operational drop state
        self.dropout = nn.Dropout(p=0.3)
        # Layer 4: Fully Connected hidden unit tier containing 128 elements
        # Flattened structural dimension value = 32 * 12 * 12 = 4608 units
        self.fc1 = nn.Linear(32 * 12 * 12, 128)
        # Layer 5: Output allocation mapping straight onto 10 dense class boundaries
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.pool(x)
        x = self.dropout(x)
        x = x.view(-1, 32 * 12 * 12)
        x = F.relu(self.fc1(x))
        # PyTorch recommendation: Use log_softmax for high numerical training stability
        return F.log_softmax(self.fc2(x), dim=1)

model_cnn = CNNClassifier()
print("\n--- CNN Architecture Design ---")
print(model_cnn)

# 4. Direct verification tracking across operational parameter sizes
print("\n--- CNN Parameter Analysis Calculation ---")
conv_w = 32 * 1 * 5 * 5
conv_b = 32
fc1_w = 128 * (32 * 12 * 12)
fc1_b = 128
fc2_w = 10 * 128
fc2_b = 10
calculated_cnn_total = conv_w + conv_b + fc1_w + fc1_b + fc2_w + fc2_b
print(f"Conv Layer params:   {conv_w} weights + {conv_b} biases = {conv_w + conv_b}")
print(f"Fully-Connected 1:  {fc1_w} weights + {fc1_b} biases = {fc1_w + fc1_b}")
print(f"Output Dense Layer: {fc2_w} weights + {fc2_b} biases = {fc2_w + fc2_b}")
print(f"Calculated Combined Model Metrics: {calculated_cnn_total}")

total_params = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f"Verified Total Trainable Model Weights: {total_params}\n")

# 5. Define the loss and optimiser
criterion = nn.NLLLoss() # Standard choice pairing beautifully with log_softmax
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=0.001)

# 6. Track error updates and tracking arrays across training stages
epochs = 5
train_losses = []
test_losses = []
test_accuracies = []

print("--- Starting CNN Training ---")
for epoch in range(epochs):
    model_cnn.train()
    running_loss = 0.0
    for images, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model_cnn(images)
        
        # Clean loss execution without manual risky log transformations
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        
    epoch_train_loss = running_loss / len(mnist_trainset)
    train_losses.append(epoch_train_loss)
    
    # Evaluation Phase
    model_cnn.eval()
    test_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in test_dataloader:
            outputs = model_cnn(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            
    epoch_test_loss = test_loss / len(mnist_testset)
    epoch_accuracy = correct.double() / len(mnist_testset)
    test_losses.append(epoch_test_loss)
    test_accuracies.append(epoch_accuracy.item())
    
    print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {epoch_train_loss:.4f} | Validation Loss: {epoch_test_loss:.4f} | Accuracy: {epoch_accuracy:.4f}")

In [ ]:
# 7. Plot metrics trend tracking configurations
print("\n--- Visualizing Performance Graphs ---")
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss Metric Trends Across Epochs')
plt.xlabel('Epoch Count')
plt.ylabel('Loss Value')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(test_accuracies, label='Validation Accuracy', color='purple')
plt.title('Classification Accuracy Trend')
plt.xlabel('Epoch Count')
plt.ylabel('Accuracy Score')
plt.legend()
plt.show()